# Task A — Experiment report (E1–E4)

Sections:
1. Load results
2. Validate fingerprints
3. Aggregate by seed
4. Compute paired differences
5. Feature ablation plots (retraining wave)
6. Frozen logit sensitivity (E1)
7. Node-type permutation (E2)
8. Per-edge-type metrics (E3)
9. Relation ablation matrix (E4)
10. HCR comparison (E5 — later)
11. Final tables

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs'

def aggregate_results(frame, group_columns, metric_columns):
    return (
        frame.groupby(group_columns)[metric_columns]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )

def paired_difference(frame, index_columns, profile_column, metric_column,
                      reference_profile, compared_profile):
    pivot = frame.pivot_table(
        index=index_columns,
        columns=profile_column,
        values=metric_column,
        aggfunc='first',
    )
    pivot['delta'] = pivot[reference_profile] - pivot[compared_profile]
    return pivot.reset_index()

def bootstrap_mean_ci(values, n_bootstrap=10000, seed=20260731):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    bootstrap_means = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    return float(np.percentile(bootstrap_means, 2.5)), float(np.percentile(bootstrap_means, 97.5))

def summarize_deltas(deltas):
    deltas = np.asarray(deltas, dtype=float)
    lo, hi = bootstrap_mean_ci(deltas)
    return {
        'mean': float(np.mean(deltas)),
        'std': float(np.std(deltas, ddof=1)) if len(deltas) > 1 else float('nan'),
        'n_pos': int((deltas > 0).sum()),
        'n': int(len(deltas)),
        'min': float(np.min(deltas)),
        'max': float(np.max(deltas)),
        'ci95': (lo, hi),
    }

## 1–2. Load results & fingerprint checks

In [ ]:
e1 = pd.read_csv(OUT / 'taskA_frozen_feature_ablation.csv') if (OUT / 'taskA_frozen_feature_ablation.csv').exists() else None
e2 = pd.read_csv(OUT / 'taskA_node_type_permutation.csv') if (OUT / 'taskA_node_type_permutation.csv').exists() else None
e3 = pd.read_csv(OUT / 'taskA_per_edge_type.csv') if (OUT / 'taskA_per_edge_type.csv').exists() else None
print('E1', None if e1 is None else e1.shape)
print('E2', None if e2 is None else e2.shape)
print('E3', None if e3 is None else e3.shape)

if e1 is not None:
    print('E1 candidate fingerprints:', sorted(e1['candidate_fingerprint'].dropna().unique()))
    print('E1 graph fingerprints:', sorted(e1['graph_fingerprint'].dropna().unique()))

## 6. Frozen logit sensitivity (E1)

In [ ]:
if e1 is not None:
    valid = e1[e1['split'] == 'valid'].copy()
    emp = valid[valid['profile'] == 'empirical'][['model', 'training_seed', 'auprc', 'brier']].rename(
        columns={'auprc': 'auprc_emp', 'brier': 'brier_emp'}
    )
    rows = []
    for pert in ['topology_only', 'empirical_shuffled']:
        sub = valid[valid['profile'] == pert][
            ['model', 'training_seed', 'auprc', 'brier', 'mean_abs_logit_change', 'spearman_logits']
        ].rename(columns={
            'auprc': 'auprc_pert',
            'brier': 'brier_pert',
            'mean_abs_logit_change': 'mean_abs_logit_change',
            'spearman_logits': 'spearman_logits',
        })
        m = emp.merge(sub, on=['model', 'training_seed'])
        m['delta_auprc'] = m['auprc_emp'] - m['auprc_pert']
        m['delta_brier'] = m['brier_pert'] - m['brier_emp']
        m['perturbation'] = pert
        rows.append(m)
    report = pd.concat(rows, ignore_index=True)
    display(
        report.groupby(['model', 'perturbation'])[
            ['delta_auprc', 'delta_brier', 'mean_abs_logit_change', 'spearman_logits']
        ].agg(['mean', 'std'])
    )

## 7. Node-type permutation (E2)

In [ ]:
if e2 is not None:
    valid = e2[e2['split'] == 'valid'].copy()
    pivot = (
        valid.groupby(['model', 'node_type'])['delta_auprc']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    display(pivot)
    for model_name, g in pivot.groupby('model'):
        g = g.sort_values('mean')
        plt.figure(figsize=(8, 4))
        plt.bar(g['node_type'], g['mean'], yerr=g['std'], capsize=4)
        plt.xticks(rotation=30, ha='right')
        plt.ylabel('mean Δ AUPRC (emp − shuffle(type))')
        plt.title(f'E2 node-type permutation — {model_name}')
        plt.tight_layout()
        plt.show()

## 8. Per-edge-type metrics (E3)

In [ ]:
if e3 is not None:
    valid = e3[(e3['split'] == 'valid') & (~e3['insufficient_support'].astype(bool))].copy()
    heat = valid.pivot_table(
        index='edge_type',
        columns=['model', 'profile'],
        values='auprc',
        aggfunc='mean',
    )
    display(heat.head(20))
    # Δ empirical − topology
    emp = valid[valid['profile'] == 'empirical'][['model', 'training_seed', 'edge_type', 'auprc']].rename(columns={'auprc': 'emp'})
    top = valid[valid['profile'] == 'topology_only'][['model', 'training_seed', 'edge_type', 'auprc']].rename(columns={'auprc': 'top'})
    d = emp.merge(top, on=['model', 'training_seed', 'edge_type'])
    d['delta'] = d['emp'] - d['top']
    display(d.groupby(['model', 'edge_type'])['delta'].agg(['mean', 'std']).sort_values('mean', ascending=False).head(15))

## 9. Relation ablation matrix (E4)

Pull W&B `TaskA_RELATION_ABLATION` runs (or local CSVs once exported) and build
`M_ij = AUPRC_j(full) − AUPRC_j(remove i)` after adding per-relation eval logging.

In [ ]:
# Placeholder: load relation-ablation summary CSV when available.
rel_path = OUT / 'taskA_relation_ablation_summary.csv'
if rel_path.exists():
    rel = pd.read_csv(rel_path)
    display(rel.head())
else:
    print('Relation ablation summary not found yet:', rel_path)